# Волна 2: настройка LightGBM в ансамбле

База - "v5 + LightGBM" (`05_lgbm`: -0.0014 / -0.0031 / -0.0021). Параметры LightGBM (`src/lgbm.py`) поставлены на глаз: 400 деревьев, lr 0.05, `num_leaves` 63, `min_data_in_leaf` 50. Пробуем, меняя только LightGBM-участника (CatBoost-участники те же, из кэша): листьев меньше (31) и больше (127), крупнее листья (`min_data_in_leaf` 200), больше деревьев с меньшим шагом (800 при lr 0.03).

Вариант проходит, если он не хуже базы (посчитанной в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import pandas as pd
import core
from core import *
import classifier, regressor, pipeline

NAME = "05_lgbm2"
# быстрее, чем в 03/04: на Kaggle - каждая 2-я неделя отсчёта (так валидировался v5); деревьев - по 400, как в модулях
if ON_KAGGLE:
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
V5 = {"threshold": 0.5, "integer": "smape", "ensemble": [{}, {"random_seed": 7}]}
LGB = {"library": "lightgbm"}
V5L = {**V5, "ensemble": [{}, {"random_seed": 7}, LGB]}                                 # v5 + LightGBM (05_lgbm)
SEASON = ["season2025", "season2024"]
sales, stock = load_data()
classifier.PARAMS, regressor.PARAMS, classifier.HALF_LIFE, classifier.ORIGIN_STEP

({'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'Logloss',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'one_hot_max_size': 64,
  'task_type': 'GPU'},
 {'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'RMSE',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'task_type': 'GPU'},
 26,
 2)

## Варианты

Параметры метода `pipeline`; база для сравнения - "v5 + LightGBM".

In [2]:
BASE_NAME = "v5 + LightGBM"
VARIANTS = {"v5 + LightGBM": V5L}
for name, lgbm_params in (("num_leaves 31", {"num_leaves": 31}), ("num_leaves 127", {"num_leaves": 127}),
                          ("min_data_in_leaf 200", {"min_data_in_leaf": 200}),
                          ("800 деревьев, lr 0.03", {"rounds": 800, "learning_rate": 0.03})):
    VARIANTS[f"LightGBM {name}"] = {**V5, "ensemble": [{}, {"random_seed": 7}, {**LGB, "lgbm": lgbm_params}]}
list(VARIANTS)

['v5 + LightGBM',
 'LightGBM num_leaves 31',
 'LightGBM num_leaves 127',
 'LightGBM min_data_in_leaf 200',
 'LightGBM 800 деревьев, lr 0.03']

## Сетка на трёх фолдах

Модели учатся один раз на фолд и набор настроек, варианты перебираются поверх кэша. Если вариант не посчитается, он пропускается - остальное считается.

In [3]:
rows = []
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    for name, params in VARIANTS.items():
        try:
            pred = make_forecast(s_cut, st_cut, "pipeline", pairs=pairs, **params)
        except Exception as e:
            print(f"{fold}, {name}: пропущен - {type(e).__name__}: {e}", flush=True)
            continue
        m = compute_metrics(truth, pred)
        rows.append({"вариант": name, "фолд": fold, **m, "score": score_minmax(m, base),
                     "прогноз / факт": pred.quantity.sum() / truth.quantity.sum()})
    print(fold, "готов", flush=True)
grid = pd.DataFrame(rows)
grid.to_csv(OUT_DIR / f"grid_{NAME}.csv", index=False)
grid.pivot_table(index="вариант", columns="фолд", values=["smape", "wape", "bias", "rmse"]).round(2)

[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2025 готов


... (4 outputs left out)

[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2024 готов


recent готов


bias                         rmse             \
фолд                           recent season2024 season2025 recent season2024   
вариант                                                                         
LightGBM 800 деревьев, lr 0.03  75.79      86.47      80.51   3.34       1.89   
LightGBM min_data_in_leaf 200   75.86      86.57      80.51   3.34       1.89   
LightGBM num_leaves 127         75.68      86.27      80.47   3.34       1.89   
LightGBM num_leaves 31          76.19      86.69      80.79   3.34       1.90   
v5 + LightGBM                   75.83      86.58      80.56   3.34       1.88   

                                           smape                         wape  \
фолд                           season2025 recent season2024 season2025 recent   
вариант                                                                         
LightGBM 800 деревьев, lr 0.03       2.65  58.60      42.88      57.46  46.98   
LightGBM min_data_in_leaf 200        2.66  58.65      42.76      57.46  46.97   
LightGBM num_leaves 127              2.66  58.65      42.91      57.53  46.98   
LightGBM num_leaves 31               2.67  58.68      42.84      57.54  47.05   
v5 + LightGBM                        2.66  58.65      42.86      57.49  47.01   

                                                      
фолд                           season2024 season2025  
вариант                                               
LightGBM 800 деревьев, lr 0.03      62.62      51.10  
LightGBM min_data_in_leaf 200       62.63      51.13  
LightGBM num_leaves 127             62.67      51.16  
LightGBM num_leaves 31              62.79      51.28  
v5 + LightGBM                       62.61      51.21

## Выбор

Вариант проходит, если он не хуже базы (посчитанной в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc[BASE_NAME]
scores["по сезону"] = scores[SEASON].mean(axis=1)
scores["проходит"] = (scores[SEASON] <= BASE[SEASON]).all(axis=1) & (scores["recent"] <= BASE["recent"] + 0.005)
scores.loc[BASE_NAME, "проходит"] = False
passed = scores[scores["проходит"]]
BEST = passed["по сезону"].idxmin() if len(passed) else None
print(BASE_NAME, "в этом запуске:", BASE[list(FOLDS)].round(4).to_dict(), "| по сезону", round(BASE[SEASON].mean(), 4))
print("лучший вариант:", BEST, scores.loc[BEST, list(FOLDS) + ["по сезону"]].astype(float).round(4).to_dict() if BEST else "- не прошёл ни один")
scores.sort_values("по сезону").round(4)

v5 + LightGBM в этом запуске: {'season2025': 0.8091, 'season2024': 0.8203, 'recent': 0.7855} | по сезону 0.8147
лучший вариант: LightGBM min_data_in_leaf 200 {'season2025': 0.8084, 'season2024': 0.82, 'recent': 0.7851, 'по сезону': 0.8142}


фолд,season2025,season2024,recent,по сезону,проходит
вариант,,,,,
LightGBM min_data_in_leaf 200,0.8084,0.8200,0.7851,0.8142,True
"LightGBM 800 деревьев, lr 0.03",0.8080,0.8207,0.7850,0.8144,False
v5 + LightGBM,0.8091,0.8203,0.7855,0.8147,False
LightGBM num_leaves 127,0.8088,0.8211,0.7850,0.8150,False
LightGBM num_leaves 31,0.8102,0.8216,0.7861,0.8159,False
